# Prepare the 2015?16 London respondent extract

This notebook reads the Year 1 SPSS file, keeps respondents in London except the City of London, selects the requested variables, and adds `year = 2016`.

Activity variables are selected by scanning the original SPSS variable names. A variable is retained when its name contains both a requested prefix and one of the 125 stable activity suffixes. The two overall measures, `MEMS7_ALL` and `MEMS7GR_ALL`, are also retained.

The notebook writes two files:

- `active_lives_1516_london_125.csv` ? respondent-level data
- `active_lives_1516_london_125_variables.csv` ? variable dictionary

The original SAV is read only. No SAV file is created.

## 1. Imports

In [1]:
from pathlib import Path

import pandas as pd
import pyreadstat
from openpyxl import load_workbook

## 2. File paths and selection rules

Run the notebook from the project folder. If an earlier output CSV exists, it is loaded before processing so that the final section can report exactly what changed.

In [2]:
project_dir = Path.cwd()

source_sav = (
    project_dir
    / "spss"
    / "spss28"
    / "active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav"
)
composite_workbook = project_dir / "eight_year_composites.xlsx"

data_file = project_dir / "active_lives_1516_london_125.csv"
variable_file = project_dir / "active_lives_1516_london_125_variables.csv"

activity_prefixes = ("MEMS7_", "MEMS7GR_", "DAYS10P60GR_", "MONTHS_12_")
overall_activity_names = ("MEMS7_ALL", "MEMS7GR_ALL")

if not source_sav.exists():
    raise FileNotFoundError(source_sav)
if not composite_workbook.exists():
    raise FileNotFoundError(composite_workbook)

# Keep the previous output in memory for the comparison at the end.
previous_data = pd.read_csv(data_file) if data_file.exists() else None
if previous_data is not None:
    print(f"Previous output loaded: {previous_data.shape}")

Previous output loaded: (19887, 531)


## 3. Read the list of stable composites

Only the 125 suffixes on the `Stable composites` sheet are used. The activity label is retained for the variable dictionary.

In [3]:
workbook = load_workbook(composite_workbook, read_only=True, data_only=True)
stable_sheet = workbook["Stable composites"]

stable_composites = [
    {"suffix": str(row[0]).strip(), "label": str(row[1]).strip()}
    for row in stable_sheet.iter_rows(min_row=2, values_only=True)
    if row[0]
]

if len(stable_composites) != 125:
    raise ValueError(
        f"Expected 125 stable composites, found {len(stable_composites)}"
    )

print(f"Stable composites: {len(stable_composites)}")

Stable composites: 125


## 4. Build the variable list

Core variables are matched without regard to case, while their original SPSS spelling is kept. Activity variables are not constructed from exact names. Instead, every original source name is scanned and retained when it contains at least one requested prefix and at least one stable activity suffix.

This rule can retain additional derived fields such as `MEMS7_CAPPED_...`. The two overall activity measures are added explicitly because their names do not contain an activity suffix.

In [4]:
_, sav_metadata = pyreadstat.read_sav(source_sav, metadataonly=True)
source_columns = sav_metadata.column_names
source_name = {name.lower(): name for name in source_columns}

core_spec = [
    ("Other", "serial"),
    ("Other", "mode"),
    ("Other", "Month"),
    ("Geography", "LA_2023"),
    ("Geography", "Reg9"),
    ("Geography", "LondInOut"),
    ("Demographics", "age16plus"),
    ("Demographics", "Age9"),
    ("Demographics", "Disab3"),
]
core_spec.extend(
    ("Demographics", f"disty{number}_POP") for number in range(1, 14)
)

core_variables = []
for category, requested_name in core_spec:
    matched_name = source_name.get(requested_name.lower())
    if matched_name is None:
        raise KeyError(f"Variable not found in the SAV: {requested_name}")
    core_variables.append({"category": category, "name": matched_name})

weight_variables = [
    name for name in source_columns if name.lower().startswith("wt_")
]

overall_activity_variables = []
for requested_name in overall_activity_names:
    matched_name = source_name.get(requested_name.lower())
    if matched_name is None:
        raise KeyError(f"Overall activity variable not found: {requested_name}")
    overall_activity_variables.append(matched_name)

activity_variables = []
for variable_name in source_columns:
    lower_name = variable_name.lower()
    matched_prefixes = [
        prefix for prefix in activity_prefixes if prefix.lower() in lower_name
    ]
    matched_composites = [
        item for item in stable_composites
        if item["suffix"].lower() in lower_name
    ]
    if matched_prefixes and matched_composites:
        activity_variables.append(
            {
                "name": variable_name,
                "prefixes": matched_prefixes,
                "suffixes": [item["suffix"] for item in matched_composites],
                "labels": [item["label"] for item in matched_composites],
            }
        )

activity_specific_names = [item["name"] for item in activity_variables]
activity_name_set = set(overall_activity_variables + activity_specific_names)
all_activity_variables = [
    name for name in source_columns if name in activity_name_set
]

activity_summary = []
for prefix in activity_prefixes:
    matched_names = [
        item["name"] for item in activity_variables if prefix in item["prefixes"]
    ]
    activity_summary.append(
        {
            "prefix": prefix,
            "source variables retained": len(matched_names),
            "examples": ", ".join(matched_names[:4]),
        }
    )

print(f"Core variables: {len(core_variables)}")
print(f"Weight variables: {len(weight_variables)}")
print(f"Activity-specific variables: {len(activity_specific_names)}")
print(f"Overall activity variables: {overall_activity_variables}")
display(pd.DataFrame(activity_summary))

Core variables: 22
Weight variables: 8
Activity-specific variables: 500
Overall activity variables: ['MEMS7_ALL', 'MEMS7GR_ALL']


,prefix,source variables retained,examples
0,MEMS7_,128,"MEMS7_CAPPED_WALKTRAV_B02, MEMS7_WALKTRAV_B02,..."
1,MEMS7GR_,124,"MEMS7GR_WALKTRAV_B02, MEMS7GR_CYCTRAV_B04, MEM..."
2,DAYS10P60GR_,124,"DAYS10P60GR_WALKTRAV_B02, DAYS10P60GR_CYCTRAV_..."
3,MONTHS_12_,124,"MONTHS_12_WALKTRAV_B02, MONTHS_12_CYCTRAV_B04,..."


## 5. Read the selected source columns

User-defined SPSS missing values are kept as their original numeric codes. Every selected activity field is an actual variable from the original SAV; no blank activity placeholders are created.

In [5]:
source_columns_to_read = (
    [item["name"] for item in core_variables]
    + weight_variables
    + all_activity_variables
)

if len(source_columns_to_read) != len(set(source_columns_to_read)):
    raise ValueError("The selected source column list contains duplicates")

source_data, selected_metadata = pyreadstat.read_sav(
    source_sav,
    usecols=source_columns_to_read,
    apply_value_formats=False,
    formats_as_category=False,
    user_missing=True,
)

print(f"Source respondents: {len(source_data):,}")
print(f"Source columns read: {len(source_data.columns):,}")

Source respondents: 198,911
Source columns read: 532


## 6. Keep London respondents and remove the City of London

London local authority labels begin with `E09`. The City of London is identified by the exact ONS code `E09000001` and removed from the allowed set before filtering.

In [6]:
la_labels = selected_metadata.variable_value_labels["LA_2023"]
all_london_codes = {
    value for value, label in la_labels.items() if str(label).startswith("E09")
}
city_of_london_codes = {
    value for value, label in la_labels.items()
    if str(label).startswith("E09000001 ")
}
london_codes = all_london_codes - city_of_london_codes

if len(all_london_codes) != 33:
    raise ValueError(f"Expected 33 London LA codes, found {len(all_london_codes)}")
if len(city_of_london_codes) != 1:
    raise ValueError(
        f"Expected one City of London code, found {len(city_of_london_codes)}"
    )
if len(london_codes) != 32:
    raise ValueError(
        f"Expected 32 London LA codes after exclusion, found {len(london_codes)}"
    )

city_of_london_rows = source_data["LA_2023"].isin(city_of_london_codes).sum()
london_data = source_data.loc[source_data["LA_2023"].isin(london_codes)].copy()

print(f"City of London respondents excluded: {city_of_london_rows:,}")
print(f"London respondents retained: {len(london_data):,}")

City of London respondents excluded: 267
London respondents retained: 19,620


## 7. Add the year and arrange the columns

`year` is inserted after `serial`. Activity variables remain in their original SPSS order after the core and weight variables.

In [7]:
ordered_columns = (
    [item["name"] for item in core_variables]
    + weight_variables
    + all_activity_variables
)
london_data = london_data.loc[:, ordered_columns]
london_data.insert(1, "year", 2016)

print("First columns:", list(london_data.columns[:6]))
print(f"Final columns: {len(london_data.columns):,}")

First columns: ['serial', 'year', 'mode', 'Month', 'LA_2023', 'Reg9']
Final columns: 533


## 8. Create the variable dictionary

The dictionary follows the exact output order. For activity-specific fields, it records the prefix and stable activity suffix that caused the original SPSS variable name to be retained.

In [8]:
dictionary_rows = []
activity_by_name = {item["name"]: item for item in activity_variables}
overall_activity_set = set(overall_activity_variables)

for item in core_variables:
    dictionary_rows.append(
        {
            "category": item["category"],
            "variable": item["name"],
            "activity_suffix": "",
            "activity_label": "",
            "source_status": "source variable",
            "selection_note": "",
        }
    )

dictionary_rows.insert(
    1,
    {
        "category": "Other",
        "variable": "year",
        "activity_suffix": "",
        "activity_label": "",
        "source_status": "constant value: 2016",
        "selection_note": "",
    },
)

for variable_name in weight_variables:
    dictionary_rows.append(
        {
            "category": "Weight",
            "variable": variable_name,
            "activity_suffix": "",
            "activity_label": "",
            "source_status": "source variable",
            "selection_note": "",
        }
    )

for variable_name in all_activity_variables:
    if variable_name in overall_activity_set:
        dictionary_rows.append(
            {
                "category": "Activity",
                "variable": variable_name,
                "activity_suffix": "",
                "activity_label": "Overall activity measure",
                "source_status": "source variable",
                "selection_note": "Added explicitly",
            }
        )
        continue

    match = activity_by_name[variable_name]
    dictionary_rows.append(
        {
            "category": "Activity",
            "variable": variable_name,
            "activity_suffix": ", ".join(match["suffixes"]),
            "activity_label": ", ".join(match["labels"]),
            "source_status": "source variable",
            "selection_note": (
                "Matched prefix(es): " + ", ".join(match["prefixes"])
            ),
        }
    )

variable_dictionary = pd.DataFrame(dictionary_rows)

## 9. Validate the extract before writing files

These checks confirm that the City of London has been removed, both overall activity measures are present, and the activity fields exactly match the source-name scanning rule.

In [9]:
assert london_data.columns[1] == "year"
assert london_data["year"].eq(2016).all()
assert london_data["LA_2023"].isin(london_codes).all()
assert not london_data["LA_2023"].isin(city_of_london_codes).any()
assert london_data["serial"].notna().all()
assert london_data["serial"].is_unique
assert variable_dictionary["variable"].tolist() == london_data.columns.tolist()

for variable_name in overall_activity_variables:
    assert variable_name in london_data.columns

expected_activity_names = {
    name
    for name in source_columns
    if any(prefix.lower() in name.lower() for prefix in activity_prefixes)
    and any(
        item["suffix"].lower() in name.lower()
        for item in stable_composites
    )
}
assert set(activity_specific_names) == expected_activity_names
assert set(all_activity_variables) == expected_activity_names | set(
    overall_activity_variables
)

print(f"Validated respondents: {len(london_data):,}")
print(f"Validated columns: {len(london_data.columns):,}")
print(f"Inner London: {(london_data['LondInOut'] == 1).sum():,}")
print(f"Outer London: {(london_data['LondInOut'] == 2).sum():,}")

Validated respondents: 19,620
Validated columns: 533
Inner London: 7,529
Outer London: 12,091


## 10. Write the two CSV files

In [10]:
london_data.to_csv(data_file, index=False, encoding="utf-8-sig")
variable_dictionary.to_csv(variable_file, index=False, encoding="utf-8-sig")

print(f"Data file: {data_file}")
print(f"Variable file: {variable_file}")

Data file: Y:\afinal\UKDA-8223-spss\active_lives_1516_london_125.csv
Variable file: Y:\afinal\UKDA-8223-spss\active_lives_1516_london_125_variables.csv


## 11. Check the written files and report changes

The written files are checked first. If a previous output was loaded at the start, the notebook then reports respondent and variable changes and verifies values for all shared respondents and columns.

In [11]:
written_data = pd.read_csv(data_file)
written_dictionary = pd.read_csv(variable_file)

assert len(written_data) == len(london_data)
assert written_data["year"].eq(2016).all()
assert written_data["LA_2023"].isin(london_codes).all()
assert not written_data["LA_2023"].isin(city_of_london_codes).any()
assert len(written_dictionary) == len(london_data.columns)

print(
    f"Final data shape: {len(written_data):,} rows x "
    f"{len(written_data.columns):,} columns"
)
print(f"Variable dictionary rows: {len(written_dictionary):,}")

if previous_data is not None:
    previous_columns = previous_data.columns.tolist()
    current_columns = written_data.columns.tolist()
    added_columns = [name for name in current_columns if name not in previous_columns]
    removed_columns = [name for name in previous_columns if name not in current_columns]

    previous_ids = set(previous_data["serial"])
    current_ids = set(written_data["serial"])
    shared_ids = previous_ids & current_ids
    removed_ids = previous_ids - current_ids
    added_ids = current_ids - previous_ids

    shared_columns = [
        name for name in previous_columns
        if name in written_data.columns and name != "serial"
    ]
    previous_aligned = (
        previous_data[previous_data["serial"].isin(shared_ids)]
        .set_index("serial")
        .sort_index()
        .loc[:, shared_columns]
    )
    current_aligned = (
        written_data[written_data["serial"].isin(shared_ids)]
        .set_index("serial")
        .sort_index()
        .loc[:, shared_columns]
    )

    shared_values_match = False
    try:
        pd.testing.assert_frame_equal(
            previous_aligned,
            current_aligned,
            check_dtype=False,
            check_exact=False,
            rtol=1e-12,
            atol=1e-12,
        )
        shared_values_match = True
    except AssertionError:
        pass

    comparison = pd.DataFrame(
        {
            "measure": [
                "Rows",
                "Columns",
                "Added respondents",
                "Removed respondents",
                "Shared respondents",
                "Added columns",
                "Removed columns",
                "Shared values match",
            ],
            "previous output": [
                len(previous_data),
                len(previous_columns),
                "",
                "",
                "",
                "",
                "",
                "",
            ],
            "revised output": [
                len(written_data),
                len(current_columns),
                len(added_ids),
                len(removed_ids),
                len(shared_ids),
                len(added_columns),
                len(removed_columns),
                shared_values_match,
            ],
        }
    )
    display(comparison)
    print("Added columns:", added_columns)
    print("Removed columns:", removed_columns)

Final data shape: 19,620 rows x 533 columns
Variable dictionary rows: 533


,measure,previous output,revised output
0,Rows,19887,19620
1,Columns,531,533
2,Added respondents,,0
3,Removed respondents,,267
4,Shared respondents,,19620
5,Added columns,,6
6,Removed columns,,4
7,Shared values match,,True


Added columns: ['MEMS7_ALL', 'MEMS7GR_ALL', 'MEMS7_CAPPED_WALKTRAV_B02', 'MEMS7_CAPPED_CYCTRAV_B04', 'MEMS7_CAPPED_ACTTRAV_C03', 'MEMS7_CAPPED_FOOTBALL_F01']
Removed columns: ['MEMS7_HULAHOOP_P27', 'MEMS7GR_HULAHOOP_P27', 'DAYS10P60GR_HULAHOOP_P27', 'MONTHS_12_HULAHOOP_P27']
